In [1]:
import pandas as pd
import os 
from sqlalchemy import create_engine
import logging
import time

logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s -%(levelname)s-%(message)s",
    filemode="a"
)


engine = create_engine('sqlite:///inventory.db')


def ingest_db(df, table_name, engine):
    '''this function will ingest the dataframe into database table'''
    df.to_sql(table_name , con=engine , if_exists = 'append' , index=False)



def load_raw_data():
    '''this function will load the csv as dataframe and ingest into db '''
    start = time.time()

    chunk_size = 50000

    logging.info("Raw data loading started")

    for file in os.listdir('New_file'):

        if file.endswith('.csv'):

            try:

                file_path = os.path.join('New_file', file)

                logging.info(f"Processing started for file : {file}")

                for i, chunk in enumerate(pd.read_csv(file_path,chunksize=chunk_size)):

                    logging.debug(
                        f"Chunk {i+1} loaded "
                        f"with shape {chunk.shape}"
                    )

                    ingest_db(chunk, file[:-4], engine)

                    logging.debug(
                        f"Chunk {i+1} inserted into table "
                        f"{file[:-4]}"
                    )

                logging.info(f"{file} completed successfully")

            except Exception as e:

                logging.error(
                    f"Error occurred while processing "
                    f"{file} : {str(e)}"
                )
    end = time.time()
    total_time=(end-start)/60
    logging.info("Raw data loading completed")

    logging.info(f'\nTotal time taken: {total_time} minutes')
if __name__== '__main__':
    load_raw_data()


In [5]:
for file in os.listdir('New_file'):
    print(file)

begin_inventory.csv
end_inventory.csv
purchases.csv
purchase_prices.csv
sales.csv
vendor_invoice.csv
